In [4]:
from multiprocess import Pool
import itertools
import json
from tqdm import tqdm
from glob import glob
from collections import defaultdict
import random
import os

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
files = glob('Emilia-YODAS/*/*.json')
len(files)

13011598

In [12]:
audio = defaultdict(list)
for f in tqdm(files):
    a = '_'.join(f.split('_')[:-1])
    audio[a].append(f)

100%|██████████| 13011598/13011598 [00:08<00:00, 1462079.00it/s]


In [37]:
audio_flat = [{'k': k, 'v': v} for k, v in audio.items()]
len(audio_flat)

331717

In [40]:
audio_flat[0]['k']

'Emilia-YODAS/ZH/ZH_o9YceLVKe2Y'

In [41]:
def loop(rows):
    rows, _ = rows
    for r in tqdm(rows):
        data = []
        for r_ in r['v']:
            with open(r_) as fopen:
                d = json.load(fopen)
            d['audio'] = r_.replace('.json', '.mp3')
            data.append(d)
        with open(r['k'] + '_combined.json', 'w') as fopen:
            json.dump(data, fopen)

In [42]:
loop((audio_flat[:1], 0))

100%|██████████| 1/1 [00:00<00:00, 68.17it/s]


In [49]:
multiprocessing(audio_flat, loop, cores = 20, returned = False)

100%|██████████| 16585/16585 [13:06<00:00, 21.09it/s]


In [50]:
combined = glob('Emilia-YODAS/*/*_combined.json')
len(combined)

331717

In [58]:
combined[-100000]

'Emilia-YODAS/EN/EN_eOIUf73q_no_combined.json'

In [62]:
with open(combined[-100001]) as fopen:
    d = json.load(fopen)
len(d)

457

In [63]:
d[0]

{'text': " There's a part of me I just like in the future that I could say I live off Godagin because Godagin was a cool street in my mind because of the song and Lady Hammond sounds like freaking awful and so it'd be nice if they just called the whole damn street Godagin because it's super confusing to be on Godagin and then all of a sudden for no apparent reason",
 'duration': 17.96,
 'speaker': 'EN_0uoizKNJReY_SPEAKER_06',
 'language': 'en',
 'dnsmos': 3.3367,
 'phone_count': 303,
 '_id': 'EN_0uoizKNJReY_W000519',
 'audio': 'Emilia-YODAS/EN/EN_0uoizKNJReY_W000519.mp3'}

In [64]:
import IPython.display as ipd
ipd.Audio(d[0]['audio'])

In [2]:
files = glob('Emilia-YODAS/*/*_combined.json')
len(files)

331717

In [3]:
with open('combined-glob.json', 'w') as fopen:
    json.dump(files, fopen)